# 01. Exploratory Data Analysis & Data Cleaning Pipeline
### Project: Smart Public Safety Analytics
**Focus:** Aggregate historical incident patterns, data quality inspection, and feature engineering.

> **CRITICAL METHODOLOGICAL NOTICE:**
> Reported Incidents ≠ Actual Underlying Crime. This notebook analyzes historical public open-data records. It does NOT predict individual behavior.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.data_loader import load_sample_dataset
from src.data_cleaning import inspect_raw_data, clean_and_validate_data
from src.feature_engineering import engineer_features
from src.eda import calculate_kpis, get_category_counts, get_trend_by_year, get_trend_by_month


## 1. Load Sample Incident Dataset & Inspect Health


In [ ]:
raw_df, mapping, missing_critical, pii_flags = load_sample_dataset()
print(f"Loaded Raw Dataset: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
print(f"PII Scan Result: {pii_flags if pii_flags else 'Safe - No PII detected'}")

raw_audit = inspect_raw_data(raw_df)
print(f"Duplicate records: {raw_audit['duplicate_count']}")
print(f"Overall missing rate: {raw_audit['overall_missing_pct']}%")
raw_df.head()


## 2. Execute Data Cleaning & Validation Pipeline
Steps:
1. Deduplication
2. Safe categorical missing imputation
3. Geographic coordinate boundary validation (-90..90, -180..180, excluding (0,0) null-island)
4. Datetime parsing and normalization


In [ ]:
cleaned_df, clean_log = clean_and_validate_data(raw_df, mapping)
print("Pipeline Cleaning Audit:")
for step, val in clean_log.items():
    print(f" - {step}: {val}")


## 3. Analytical Feature Engineering
Deriving diurnal time periods (Night, Morning, Afternoon, Evening), calendar features, and weekend flags.


In [ ]:
featured_df = engineer_features(cleaned_df)
print("Engineered Columns:", [c for c in featured_df.columns if c not in cleaned_df.columns])
featured_df[["incident_id", "date", "time", "crime_category", "time_period", "is_weekend"]].head()


## 4. Exploratory Data Distributions


In [ ]:
kpis = calculate_kpis(featured_df, raw_audit)
print("Summary KPIs:", kpis)

top_crimes = get_category_counts(featured_df, top_n=8)
print("\nTop Reported Crime Categories:\n", top_crimes)
